# 01_data_inspect_and_metadata
- Inspect species, disease, DeepFish folders
- Build metadata_raw.csv
- Map disease labels to freshness proxy (0..1)


In [32]:
# 01_Final_Metadata.py
# Robust Notebook 01 replacement: Inspect dataset, build metadata with species, disease, deepfish
# Run cells in order in a Jupyter/IPython notebook environment.
# TL-style emojis sprinkled for morale.

# Cell 1: header (markdown in notebook)
# """
# � 01 — Dataset Inspect & Metadata Forge �✨
# This script inspects your raw datasets (species, disease, DeepFish) and builds
# metadata_rebuilt_raw.csv and metadata_rebuilt_mapped.csv in data_sih/processed/.
# Run top-to-bottom. If something breaks, read printed diagnostics.
# """

# Cell 2: imports & paths
from pathlib import Path
import pandas as pd
import numpy as np
import os

ROOT = Path("data_sih")
SPECIES_ROOT = ROOT / "species"
DISEASE_ROOT = ROOT / "disease"
DEEP_ROOT = ROOT / "deepfish" / "DeepFish" / "Segmentation"
PROC = ROOT / "processed"
PROC.mkdir(parents=True, exist_ok=True)

print("ROOT exists:", ROOT.exists())
print("species folder:", SPECIES_ROOT.exists(), "| disease folder:", DISEASE_ROOT.exists())
print("deepfish segmentation folder:", DEEP_ROOT.exists())
print("processed folder:", PROC)

# Cell 3: quick EDA

def folder_listing(base):
    if not base.exists(): return "MISSING"
    try:
        subs = [p.name for p in sorted(base.iterdir()) if p.is_dir()]
        return subs[:40]
    except Exception as e:
        return str(e)

print("Species top-level subfolders (sample):", folder_listing(SPECIES_ROOT))
print("Disease top-level subfolders (sample):", folder_listing(DISEASE_ROOT))
print("DeepFish Segmentation subfolders (sample):", folder_listing(DEEP_ROOT))
if DEEP_ROOT.exists():
    sample_files = list(DEEP_ROOT.rglob("*.*"))[:30]
    print("Sample files under DeepFish/Segmentation (first 30):")
    for p in sample_files[:30]:
        print(" ", p.relative_to(DEEP_ROOT))

# Cell 4: robust metadata builder
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
SKIP_KEYWORDS = ("gt","_gt","mask","masks","label","labels","annotation","annotations","__MACOSX")

def gather_images(base_path, skip_keywords=SKIP_KEYWORDS):
    imgs = []
    if not base_path.exists(): return imgs
    for p in base_path.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            skip = False
            for part in p.parts:
                low = str(part).lower()
                if any(k in low for k in skip_keywords):
                    skip = True
                    break
            if skip: 
                continue
            imgs.append(p)
    return imgs

rows = []

# species
if SPECIES_ROOT.exists():
    for sp_top in sorted([p for p in SPECIES_ROOT.iterdir() if p.is_dir()]):
        species_label = sp_top.name
        imgs = gather_images(sp_top)
        for img in imgs:
            rows.append({
                "image_path": str(img.resolve()),
                "source": "species",
                "species": species_label,
                "disease_label": np.nan,
                "mask_path": np.nan,
                "freshness": np.nan,
                "head_x": np.nan, "head_y": np.nan, "tail_x": np.nan, "tail_y": np.nan
            })
print(f"➡ Collected species rows: {sum(1 for r in rows if r['source']=='species')}")

# === FINAL DISEASE SCANNER: supports Train/ and Test/ plus top-level class folders ===
rows_disease = []

if DISEASE_ROOT.exists():
    # check Train/Test containers
    candidates = []
    for subname in ["Train","train","Test","test"]:
        p = DISEASE_ROOT / subname
        if p.exists() and p.is_dir():
            candidates.append(p)

    # scan each Train/Test folder
    if candidates:
        print("➡ Scanning disease subfolders:", [str(p.name) for p in candidates])
        for container in candidates:
            for cls in sorted([p for p in container.iterdir() if p.is_dir()]):
                disease_name = cls.name
                imgs = []
                for ext in IMG_EXTS:
                    imgs += list(cls.rglob(f"*{ext}"))
                print(f"   • {container.name}/{disease_name}: {len(imgs)} images")
                for img in imgs:
                    rows_disease.append({
                        "image_path": str(img.resolve()),
                        "source": "disease",
                        "species": "unknown",
                        "disease_label": disease_name,
                        "mask_path": np.nan,
                        "freshness": np.nan,
                        "head_x": np.nan, "head_y": np.nan, "tail_x": np.nan, "tail_y": np.nan
                    })

    # scan top-level class folders (ignore Train/Test)
    top_level = [d for d in DISEASE_ROOT.iterdir() if d.is_dir() and d.name.lower() not in ("train","test")]
    if top_level:
        print("➡ Scanning top-level disease class folders:", [d.name for d in top_level])
        for cls in sorted(top_level):
            imgs = []
            for ext in IMG_EXTS:
                imgs += list(cls.rglob(f"*{ext}"))
            print(f"   • {cls.name}: {len(imgs)} images")
            for img in imgs:
                rows_disease.append({
                    "image_path": str(img.resolve()),
                    "source": "disease",
                    "species": "unknown",
                    "disease_label": cls.name,
                    "mask_path": np.nan,
                    "freshness": np.nan,
                    "head_x": np.nan, "head_y": np.nan, "tail_x": np.nan, "tail_y": np.nan
                })

else:
    print("⚠️ No disease folder found at", DISEASE_ROOT)

print("➡ Total disease rows collected:", len(rows_disease))
rows.extend(rows_disease)

# DeepFish segmentation
if DEEP_ROOT.exists():
    possible_img_dirs = [DEEP_ROOT/"images", DEEP_ROOT/"Images", DEEP_ROOT]
    images = []
    for d in possible_img_dirs:
        if d.exists():
            for ext in IMG_EXTS:
                images += list(d.rglob(f"*{ext}"))
            if images:
                img_root = d
                break
    possible_mask_dirs = [DEEP_ROOT/"masks", DEEP_ROOT/"Masks", DEEP_ROOT/"masks_png", DEEP_ROOT]
    mask_index = {}
    for d in possible_mask_dirs:
        if d.exists():
            for p in d.rglob("*.*"):
                if p.suffix.lower() in IMG_EXTS:
                    mask_index.setdefault(p.stem.lower(), []).append(str(p.resolve()))
    for img in sorted(images):
        stem = img.stem.lower()
        mask_candidate = None
        if stem in mask_index:
            cands = mask_index[stem]
            chosen = None
            for c in cands:
                low = c.lower()
                if "mask" in low or "/masks/" in low.replace("\\","/"):
                    chosen = c; break
            if chosen is None:
                chosen = cands[0]
            mask_candidate = chosen
        rows.append({
            "image_path": str(img.resolve()),
            "source": "deepfish",
            "species": "unknown",
            "disease_label": np.nan,
            "mask_path": mask_candidate if mask_candidate else np.nan,
            "freshness": np.nan,
            "head_x": np.nan, "head_y": np.nan, "tail_x": np.nan, "tail_y": np.nan
        })
    print(f"➡ Added DeepFish images: {len(images)}")
else:
    print("⚠️ DeepFish segmentation folder not found at", DEEP_ROOT)

# finalize
df_meta = pd.DataFrame(rows).drop_duplicates(subset=["image_path"]).reset_index(drop=True)
out_raw = PROC / "metadata_rebuilt_raw.csv"
df_meta.to_csv(out_raw, index=False)
print("✅ WROTE:", out_raw)
print("Counts by source:\n", df_meta['source'].value_counts())

# Cell 5: mapping disease -> freshness
# Cell 5: mapping disease -> freshness (IMPROVED)
# --- BEGIN NEW FRESHNESS MAPPING BLOCK ---
import re

mapping_file = out_raw
df = pd.read_csv(mapping_file)

def clean_label(s):
    if pd.isna(s) or str(s).strip() == "":
        return ""
    s = str(s)
    s = s.replace('_', ' ').replace('-', ' ')
    s = re.sub(r"\.(jpg|jpeg|png|bmp|tif|tiff)$", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\(\d+\)", "", s)
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

df['disease_label_clean'] = df['disease_label'].fillna('').apply(clean_label)

keyword_map = {
    'healthy': 1.0,
    'fresh': 1.0,
    'mild': 0.85,
    'early': 0.8,
    'moderate': 0.6,
    'red': 0.5,
    'gill': 0.45,
    'fin rot': 0.45,
    'disease': 0.4,
    'bacterial': 0.2,
    'aeromoniasis': 0.2,
    'infected': 0.2,
    'parasite': 0.25,
    'parasitic': 0.25,
    'saprolegniasis': 0.2,
    'fungal': 0.2,
    'white tail': 0.2
}

def map_label_to_freshness(cln):
    if not isinstance(cln, str) or cln.strip() == "":
        return np.nan
    matches = []
    for k, v in keyword_map.items():
        if k in cln:
            matches.append((k, v))
    if not matches:
        return np.nan
    matches.sort(key=lambda x: len(x[0]), reverse=True)
    return matches[0][1]

if 'freshness' not in df.columns:
    df['freshness'] = np.nan

for idx, row in df[df['source']=='disease'].iterrows():
    cln = row['disease_label_clean']
    val = map_label_to_freshness(cln)
    if not pd.isna(val):
        df.at[idx, 'freshness'] = val

unique_clean = sorted([x for x in df['disease_label_clean'].unique() if x])
unmapped = [x for x in unique_clean if all(k not in x for k in keyword_map.keys())]
print("Distinct cleaned disease labels (sample 60):", unique_clean[:60])
print("Unmapped cleaned labels (first 40):", unmapped[:40])
print("Mapped freshness count:", df['freshness'].notna().sum())

out_mapped = PROC / "metadata_rebuilt_mapped.csv"
df.to_csv(out_mapped, index=False)
print("✅ WROTE:", out_mapped)
print("Counts by source:", df['source'].value_counts())
print("Rows with freshness labels:", df['freshness'].notna().sum())
# --- END NEW FRESHNESS MAPPING BLOCK ---


# Cell 7: final quick checks
print("Final quick checks:")
print("Total rows:", len(df))
print(df['source'].value_counts())
print("Sample species labels:", sorted(df[df['source']=='species']['species'].unique())[:30])
print("Sample disease labels:", sorted(df[df['source']=='disease']['disease_label'].dropna().unique())[:30])
print("DeepFish masks present for:", df[df['source']=='deepfish']['mask_path'].notna().sum(), "images")


ROOT exists: True
species folder: True | disease folder: True
deepfish segmentation folder: True
processed folder: data_sih\processed
Species top-level subfolders (sample): ['Black Sea Sprat', 'Gilt-Head Bream', 'Hourse Mackerel', 'Red Mullet', 'Red Sea Bream', 'Sea Bass', 'Shrimp', 'Striped Red Mullet', 'Trout']
Disease top-level subfolders (sample): ['Test', 'Train']
DeepFish Segmentation subfolders (sample): ['images', 'masks']
Sample files under DeepFish/Segmentation (first 30):
  segmentation.csv
  test.csv
  train.csv
  val.csv
  images\empty\7117_no_fish_2_f000000.jpg
  images\empty\7117_no_fish_2_f000010.jpg
  images\empty\7117_no_fish_2_f000020.jpg
  images\empty\7117_no_fish_2_f000030.jpg
  images\empty\7117_no_fish_2_f000040.jpg
  images\empty\7117_no_fish_2_f000050.jpg
  images\empty\7117_no_fish_2_f000060.jpg
  images\empty\7117_no_fish_2_f000070.jpg
  images\empty\7117_no_fish_2_f000080.jpg
  images\empty\7393_NF2_f000000.jpg
  images\empty\7393_NF2_f000010.jpg
  images\e